# Final Model Training

In this notebook, we instantiate and train the final model using the optimal hyperparameters identified in the previous tuning step.

**Objectives:**
1.  **Configuration:** Set up the model with the best parameters.
2.  **Training:** Retrain the model on the full training dataset.
3.  **Evaluation:** Verify performance on the separate test set.
4.  **Deployment:** Save the trained pipeline for use in the API.

In [ ]:
# Import necessary libraries
import os
import sys
import pandas as pd
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import classification_report, roc_auc_score
from catboost import CatBoostClassifier

# Add the project root directory to sys.path to allow importing from 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import the custom preprocessor after adding path
from src.ml.preprocessor import CustomerChurnPreprocessor

In [ ]:
# Define the best hyperparameters for CatBoost
best_params_catboost = {
    'iterations': 505,
    'learning_rate': 0.025232339470106814,
    'depth': 4,
    'l2_leaf_reg': 0.4487280568776956,
    'border_count': 87,
    'subsample': 0.727617583882814,
    'random_state': 42,
    'verbose': 0,               
    'allow_writing_files': False
}

print("Starting Final Model Training (CatBoost)...")

# Load the datasets
print("Loading data...")
train_data = pd.read_csv('data/processed/train.csv')
test_data = pd.read_csv('data/processed/test.csv')
ground_truth = pd.read_csv('data/processed/ground_truth.csv')

# Preprocess the data using the custom preprocessor
preprocessor = CustomerChurnPreprocessor()

train_data_processed = preprocessor.preprocess(train_data)
test_data_processed = preprocessor.preprocess(test_data)

In [ ]:
# Separate features and target variable
X_train = train_data_processed.drop('Exited', axis=1)
y_train = train_data_processed['Exited']

X_test = test_data_processed
y_test = ground_truth['Exited']

In [ ]:
# Create the final pipeline with preprocessing and the regressor
final_pipeline = make_pipeline(
    VarianceThreshold(threshold=0),
    MinMaxScaler(),
    CatBoostClassifier(**best_params_catboost)
)

In [ ]:
# Train the model on the training data
final_pipeline.fit(X_train, y_train)

In [ ]:
# Generate predictions on the test set
y_pred = final_pipeline.predict(X_test)
y_prob = final_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Calculate classification metrics
roc_score = roc_auc_score(y_test, y_prob)

In [ ]:
print("\n" + "="*40)
print(f" FINAL ROC-AUC SCORE: {roc_score:.4f}")
print("="*40)
print(classification_report(y_test, y_pred))

In [ ]:
# Save the trained model pipeline to disk
model_dir = 'models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
model_path = os.path.join(model_dir, 'catboost_churn_model.pkl')
joblib.dump(final_pipeline, model_path)
print(f"\n Model and Pipeline successfully saved to: {model_path}")